In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset using the exact file name from your zip archive
df = pd.read_csv("GaltonFamilies.csv")
print(f"Dataset successfully loaded! Found {len(df)} records.")

Dataset successfully loaded! Found 934 records.


Part 1: Probability Distributions



In [ ]:
# ==========================================
# CELL 2: EM Algorithm & Live Prediction Engine
# ==========================================

import numpy as np

# Extract child heights
data = df['childHeight'].values

# -------------------------------
# INITIAL PARAMETERS
# -------------------------------
mu1, mu2 = 60.0, 70.0
var1, var2 = 4.0, 4.0
pi1, pi2 = 0.5, 0.5


# -------------------------------
# Gaussian PDF
# -------------------------------
def normal_pdf(x, mean, var):
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-((x - mean) ** 2) / (2 * var))


# -------------------------------
# Log-Likelihood
# -------------------------------
def log_likelihood(data, mu1, mu2, var1, var2, pi1, pi2):
    likelihood = (
        pi1 * normal_pdf(data, mu1, var1)
        +
        pi2 * normal_pdf(data, mu2, var2)
    )
    return np.sum(np.log(likelihood))


# -------------------------------
# Tracking Table Header
# -------------------------------
print(f"{'Iteration':<10}"
      f"{'Mean 1':<10}"
      f"{'Mean 2':<10}"
      f"{'Variance 1':<12}"
      f"{'Variance 2':<12}"
      f"{'Pi1':<10}"
      f"{'Pi2':<10}"
      f"{'Log-Likelihood'}")

print("-"*95)


# ==========================================
# EM LOOP
# ==========================================
for iteration in range(3):

    # -------- E STEP --------
    r1 = pi1 * normal_pdf(data, mu1, var1)
    r2 = pi2 * normal_pdf(data, mu2, var2)

    total_r = r1 + r2

    w1 = r1 / total_r
    w2 = r2 / total_r

    ll = log_likelihood(
        data,
        mu1,
        mu2,
        var1,
        var2,
        pi1,
        pi2
    )

    print(f"{iteration:<10}"
          f"{mu1:<10.2f}"
          f"{mu2:<10.2f}"
          f"{var1:<12.2f}"
          f"{var2:<12.2f}"
          f"{pi1:<10.2f}"
          f"{pi2:<10.2f}"
          f"{ll:.2f}")

    # -------- M STEP --------
    N1 = np.sum(w1)
    N2 = np.sum(w2)

    mu1 = np.sum(w1 * data) / N1
    mu2 = np.sum(w2 * data) / N2

    var1 = np.sum(w1 * (data - mu1) ** 2) / N1
    var2 = np.sum(w2 * (data - mu2) ** 2) / N2

    pi1 = N1 / len(data)
    pi2 = N2 / len(data)


# ==========================================
# LIVE PREDICTION ENGINE
# ==========================================
def predict_live_height_cm(height_input):
    """
    Accepts either:
        meters (e.g. 1.81)
        centimeters (e.g. 181)
    Converts to inches because the Galton dataset uses inches.
    """
    # Convert meters → cm
    if height_input < 3:
        cm = height_input * 100
    else:
        cm = height_input

    # Convert cm → inches
    converted_inches = cm / 2.54

    # Posterior probabilities
    p1 = pi1 * normal_pdf(converted_inches, mu1, var1)
    p2 = pi2 * normal_pdf(converted_inches, mu2, var2)

    total = p1 + p2

    prob1 = p1 / total
    prob2 = p2 / total

    print("\n" + "="*45)
    print("      LIVE ENGINE OUTPUT (GROUP 12)")
    print("="*45)
    print(f"Original Input  : {height_input}")
    print(f"Converted Height: {converted_inches:.2f} inches")
    print(f"Probability Group 1 (Shorter Cohort): {prob1*100:.2f}%")
    print(f"Probability Group 2 (Taller Cohort) : {prob2*100:.2f}%")

    if prob1 > prob2:
        print("Prediction: Group 1 (Shorter Cohort)")
    else:
        print("Prediction: Group 2 (Taller Cohort)")
    print("="*45)


# ==========================================
# PRESENTATION DAY TEST RUN
# ==========================================
# This will execute instantly when you click play!
# If the professor gives a different number, just change this value and click run again.
predict_live_height_cm(1.51)

Iteration Mean 1    Mean 2    Variance 1  Variance 2  Pi1       Pi2       Log-Likelihood
-----------------------------------------------------------------------------------------------
0         60.00     70.00     4.00        4.00        0.50      0.50      -3104.58
1         62.94     68.75     2.78        6.45        0.34      0.66      -2516.67
2         63.11     68.65     3.32        7.22        0.34      0.66      -2506.25

      LIVE ENGINE OUTPUT (GROUP 12)
Original Input  : 1.51
Converted Height: 59.45 inches
Probability Group 1 (Shorter Cohort): 96.30%
Probability Group 2 (Taller Cohort) : 3.70%
Prediction: Group 1 (Shorter Cohort)


Part 4: Gradient Descent in Code


In [ ]:
====================================================================
             GROUP 12: GRADIENT DESCENT VERIFICATION
====================================================================

--- Iteration 1 (Calculated Step) ---
  Predictions (y_hat) : [ 6. 17.]
  Error Vector (e)    : [ 1. 11.]
  Manual Gradient m   : [ 45. 113.]
  SciPy Verified Grad : [ 45.0000085  113.00005452]
  Updated Parameters  : m = [-1.45  0.87], b = [0.88 0.88]
  Current MSE Loss    : 61.0000

--- Iteration 2 (Calculated Step) ---
  Predictions (y_hat) : [2.04 3.78]
  Error Vector (e)    : [-2.96 -2.22]
  Manual Gradient m   : [-11.84 -31.08]
  SciPy Verified Grad : [-11.8399915 -31.0799455]
  Updated Parameters  : m = [-1.3316  1.1808], b = [0.9318 0.9318]
  Current MSE Loss    : 6.8450

--- Iteration 3 (Calculated Step) ---
  Predictions (y_hat) : [3.1426 7.4134]
  Error Vector (e)    : [-1.8574  1.4134]
  Manual Gradient m   : [3.7962 8.5618]
  SciPy Verified Grad : [3.7962085 8.5618545]
  Updated Parameters  : m = [-1.369562  1.095182], b = [0.93624 0.93624]
  Current MSE Loss    : 2.7238

--- Iteration 4 (Calculated Step) ---
  Predictions (y_hat) : [2.852224 6.409812]
  Error Vector (e)    : [-2.147776  0.409812]
  Manual Gradient m   : [-0.508528 -2.345208]
  SciPy Verified Grad : [-0.5085195 -2.3451535]
  Updated Parameters  : m = [-1.36447672  1.11863408], b = [0.95361964 0.95361964]
  Current MSE Loss    : 2.3904

====================================================================
                       FINAL CONVERGENCE SUMMARY
====================================================================
Final Weights (m)      : [-1.43094827  1.1122971 ]
Final Bias Vector (b)  : [1.17676457 1.17676457]
Final Model Predictions: [3.0827076  6.57594248]
True Target Vector (y) : [5. 6.]
====================================================================



SyntaxError: invalid syntax (4024635662.py, line 1)